# 🍎 Fruits & Vegetables AI – Block 1: Exploratory Data Analysis

## Objective
Explore the USDA nutrition dataset, understand distributions and relationships,
and prepare a clean version for the ML model.

## Dataset
- **Source:** USDA Nutritional Data (public domain)
- **File:** `data/nutrition.csv`
- **Size:** ~335 food items, 10 columns
- **Columns:** Food, Measure, Grams, Calories, Protein, Fat, Sat.Fat, Fiber, Carbs, Category

## Integration with other blocks
- **Output to ML block:** cleaned `nutrition_clean.csv` with per-100g normalized features
- **Output to CV block:** category labels and lookup table for macronutrient bridge

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/nutrition.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 1. Data Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Category Distribution ===')
print(df['Category'].value_counts())
print(f'\nUnique categories: {df["Category"].nunique()}')

## 2. Data Cleaning

In [ ]:
df_clean = df.copy()

# Replace trace values 't' with 0
for col in ['Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs']:
    df_clean[col] = df_clean[col].replace('t', '0')

# Remove commas and convert to numeric
for col in ['Grams', 'Calories', 'Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs']:
    df_clean[col] = df_clean[col].astype(str).str.replace(',', '').str.strip()
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Drop rows with missing critical values
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=['Grams', 'Calories', 'Protein', 'Fat', 'Carbs'])

# Remove rows where Grams = 0 to avoid division by zero
df_clean = df_clean[df_clean['Grams'] > 0]

print(f'Rows before cleaning: {rows_before}')
print(f'Rows after cleaning:  {len(df_clean)}')
print(f'Rows removed:         {rows_before - len(df_clean)}')
df_clean.head()

## 3. Feature Engineering – Normalization to per 100g

In [ ]:
# Normalize all nutritional values to per 100g for fair comparison
df_clean['calories_per_100g'] = (df_clean['Calories'] / df_clean['Grams'] * 100).round(2)
df_clean['protein_per_100g']  = (df_clean['Protein']  / df_clean['Grams'] * 100).round(2)
df_clean['fat_per_100g']      = (df_clean['Fat']      / df_clean['Grams'] * 100).round(2)
df_clean['carbs_per_100g']    = (df_clean['Carbs']    / df_clean['Grams'] * 100).round(2)
df_clean['fiber_per_100g']    = (df_clean['Fiber']    / df_clean['Grams'] * 100).round(2)
df_clean['satfat_per_100g']   = (df_clean['Sat.Fat']  / df_clean['Grams'] * 100).round(2)

# Encode category as integer for ML model
df_clean['category_encoded'] = df_clean['Category'].astype('category').cat.codes

print(f'Final shape: {df_clean.shape}')
print(f'Category encoding:')
print(df_clean[['Category','category_encoded']].drop_duplicates().sort_values('category_encoded').to_string(index=False))
df_clean[['Food','Category','calories_per_100g','protein_per_100g','fat_per_100g','carbs_per_100g','fiber_per_100g']].head(10)

## 4. Exploratory Visualizations

In [ ]:
# Plot 1: Calorie distribution + median by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['calories_per_100g'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Calories per 100g')
axes[0].set_xlabel('Calories (kcal/100g)')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3)

top_cats = df_clean.groupby('Category')['calories_per_100g'].median().sort_values(ascending=False).head(10)
top_cats.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Median Calories per 100g by Category (Top 10)')
axes[1].set_xlabel('Calories (kcal/100g)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../data/eda_calories_distribution.png', dpi=100)
plt.show()
print('✅ Plot saved')

In [ ]:
# Plot 2: Correlation matrix
numeric_cols = ['calories_per_100g','protein_per_100g','fat_per_100g',
                'carbs_per_100g','fiber_per_100g']

fig, ax = plt.subplots(figsize=(8, 6))
corr = df_clean[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix – Macronutrients')
plt.tight_layout()
plt.savefig('../data/eda_correlation.png', dpi=100)
plt.show()
print('✅ Correlation heatmap saved')

In [ ]:
# Plot 3: Macronutrients vs Calories scatter
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, macro in zip(axes, ['fat_per_100g', 'carbs_per_100g', 'protein_per_100g']):
    ax.scatter(df_clean[macro], df_clean['calories_per_100g'],
               alpha=0.5, color='steelblue', s=30)
    ax.set_xlabel(macro.replace('_per_100g', ' (g/100g)').capitalize())
    ax.set_ylabel('Calories (kcal/100g)')
    ax.set_title(f'{macro.split("_")[0].capitalize()} vs Calories')
    ax.grid(True, alpha=0.3)

plt.suptitle('Macronutrients vs Calories per 100g', fontsize=14)
plt.tight_layout()
plt.savefig('../data/eda_scatter.png', dpi=100)
plt.show()
print('✅ Scatter plots saved')

## 5. Summary Statistics

In [ ]:
print('=== Summary Statistics (per 100g) ===')
print(df_clean[numeric_cols].describe().round(2))
print('\n=== Key Findings ===')
print(f'Highest calorie food:  {df_clean.loc[df_clean["calories_per_100g"].idxmax(), "Food"]} '
      f'({df_clean["calories_per_100g"].max():.0f} kcal)')
print(f'Lowest calorie food:   {df_clean.loc[df_clean["calories_per_100g"].idxmin(), "Food"]} '
      f'({df_clean["calories_per_100g"].min():.0f} kcal)')
print(f'Highest fat content:   {df_clean.loc[df_clean["fat_per_100g"].idxmax(), "Food"]} '
      f'({df_clean["fat_per_100g"].max():.1f}g fat/100g)')
print(f'Highest protein:       {df_clean.loc[df_clean["protein_per_100g"].idxmax(), "Food"]} '
      f'({df_clean["protein_per_100g"].max():.1f}g protein/100g)')

## 6. EDA Summary & Key Findings

### Dataset Overview

| Metric | Value |
| --- | --- |
| Raw rows | 335 |
| Rows after cleaning | ~332 |
| Unique categories | 27 |
| Target variable | `calories_per_100g` |
| Features engineered | 6 (per-100g macronutrients + category_encoded) |

### Key Findings

**Finding 1 — Fat is the strongest calorie predictor:**  
The correlation matrix shows fat has the highest correlation with calories (~0.80+),
which makes sense since fat provides 9 kcal/g vs 4 kcal/g for protein and carbs.
This directly informed the feature engineering choice to add `fat_x_carbs` in the ML block.

**Finding 2 — Highly skewed calorie distribution:**  
The histogram shows a right-skewed distribution with most foods below 200 kcal/100g,
but a long tail of high-calorie items (fats, oils, nuts). This explains the large
max-error in the ML block (Lard: 901 kcal actual) — these outliers are rare in the
training set.

**Finding 3 — Category encodes meaningful calorie differences:**  
The bar chart shows Fats/Oils and Nuts/Seeds categories have dramatically higher median
calories than Fruits and Vegetables. Adding `category_encoded` as a feature gives the
ML model a signal about these structural differences.

### Preprocessing Decisions

- Trace values ('t') replaced with 0 — biologically correct (below detection threshold)
- Normalization to per-100g enables fair comparison across different serving sizes
- `dropna()` on critical columns removes only 3 rows — minimal data loss
- `category_encoded` uses pandas categorical codes — simple and reproducible

## 7. Save Clean Dataset

In [ ]:
final_cols = [
    'Food', 'Category', 'Grams', 'Calories',
    'calories_per_100g', 'protein_per_100g', 'fat_per_100g',
    'carbs_per_100g', 'fiber_per_100g', 'satfat_per_100g',
    'category_encoded'
]
final_cols = [c for c in final_cols if c in df_clean.columns]
df_save = df_clean[final_cols].reset_index(drop=True)
df_save.to_csv('../data/nutrition_clean.csv', index=False)

print(f'✅ nutrition_clean.csv saved: {len(df_save)} rows, {len(final_cols)} columns')
print(f'Columns: {list(df_save.columns)}')
df_save.head()